# 08 — Modern Architecture Components

**Companion to [Chapter 05, section 5.5](../05-transformer-architecture.md).**

GPT-2 is 2019. This notebook implements what replaced each of its components,
and measures why.

| GPT-2 | Modern | Why |
|---|---|---|
| LayerNorm | **RMSNorm** | cheaper, same quality |
| GELU FFN | **SwiGLU** | better per parameter |
| learned positions | **RoPE** | relative, no length limit |
| MHA | **GQA** | smaller KV cache |
| dense FFN | **MoE** | capacity without compute |

In [1]:
import time
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
print("torch", torch.__version__)

torch 2.13.0+cu130


## 1. RMSNorm

Drop the mean subtraction and the shift. Just scale by root-mean-square.

In [2]:
class LayerNormClassic(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.scale = nn.Parameter(torch.ones(d))
        self.shift = nn.Parameter(torch.zeros(d))
        self.eps = 1e-5

    def forward(self, x):
        mean = x.mean(-1, keepdim=True)
        var = x.var(-1, keepdim=True, unbiased=False)
        return self.scale * (x - mean) / torch.sqrt(var + self.eps) + self.shift


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(d))
        self.eps = eps

    def forward(self, x):
        rms = torch.sqrt(x.pow(2).mean(-1, keepdim=True) + self.eps)
        return self.weight * (x / rms)


d = 1024
ln, rms = LayerNormClassic(d), RMSNorm(d)
x = torch.randn(32, 512, d)

print(f"LayerNorm parameters: {sum(p.numel() for p in ln.parameters()):,}")
print(f"RMSNorm   parameters: {sum(p.numel() for p in rms.parameters()):,}   (half)")

for name, mod in [("LayerNorm", ln), ("RMSNorm", rms)]:
    t0 = time.time()
    for _ in range(20):
        mod(x)
    print(f"{name:10} {(time.time() - t0) / 20 * 1000:6.2f} ms/call")

print("\nRMSNorm skips the mean pass and the shift parameter.")
print("Quality is indistinguishable in practice - a rare free win.")

LayerNorm parameters: 2,048
RMSNorm   parameters: 1,024   (half)


LayerNorm   78.79 ms/call


RMSNorm     37.34 ms/call

RMSNorm skips the mean pass and the shift parameter.
Quality is indistinguishable in practice - a rare free win.


## 2. SwiGLU

A **gated** FFN: one branch multiplicatively filters the other, per token.

In [3]:
class FFNClassic(nn.Module):
    def __init__(self, d):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d, 4 * d), nn.GELU(), nn.Linear(4 * d, d))

    def forward(self, x):
        return self.net(x)


class SwiGLU(nn.Module):
    def __init__(self, d, hidden=None):
        super().__init__()
        hidden = hidden or int(8 * d / 3)      # ~8/3 keeps params comparable to 4x with 2 matrices
        self.gate = nn.Linear(d, hidden, bias=False)
        self.up   = nn.Linear(d, hidden, bias=False)
        self.down = nn.Linear(hidden, d, bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


d = 768
classic, swiglu = FFNClassic(d), SwiGLU(d)
print(f"GELU FFN (4x, 2 matrices) : {sum(p.numel() for p in classic.parameters()):,}")
print(f"SwiGLU  (8/3x, 3 matrices): {sum(p.numel() for p in swiglu.parameters()):,}")
print("\nComparable parameter counts. The gate gives a content-dependent filter:")
print("the model decides which hidden features pass through for THIS token.")

x = torch.randn(2, 8, d)
print("\nboth produce", tuple(classic(x).shape), "and", tuple(swiglu(x).shape))

GELU FFN (4x, 2 matrices) : 4,722,432
SwiGLU  (8/3x, 3 matrices): 4,718,592

Comparable parameter counts. The gate gives a content-dependent filter:
the model decides which hidden features pass through for THIS token.

both produce (2, 8, 768) and (2, 8, 768)


## 3. RoPE

Rotate Q and K by an angle proportional to position. Because rotation is
orthogonal, the dot product ends up depending only on the *difference* of
positions — relative position falls out of the geometry.

In [4]:
def build_rope_cache(head_dim, max_len, base=10000.0):
    inv_freq = 1.0 / (base ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(max_len).float(), inv_freq)
    angles = torch.cat([angles, angles], dim=-1)
    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):
    seq_len, head_dim = x.shape[-2], x.shape[-1]
    cos, sin = cos[:seq_len], sin[:seq_len]
    x1, x2 = x[..., :head_dim // 2], x[..., head_dim // 2:]
    rotated = torch.cat([-x2, x1], dim=-1)
    return x * cos + rotated * sin


head_dim, max_len = 64, 256
cos, sin = build_rope_cache(head_dim, max_len)
print("rope cache:", tuple(cos.shape), "- zero learned parameters")

# THE key property: identical content at different absolute positions,
# same relative distance -> same attention score
content_q = torch.randn(1, 1, 1, head_dim)
content_k = torch.randn(1, 1, 1, head_dim)

def score_at(pos_q, pos_k):
    q = torch.zeros(1, 1, max_len, head_dim); q[0, 0, pos_q] = content_q
    k = torch.zeros(1, 1, max_len, head_dim); k[0, 0, pos_k] = content_k
    qr, kr = apply_rope(q, cos, sin), apply_rope(k, cos, sin)
    return (qr[0, 0, pos_q] @ kr[0, 0, pos_k]).item()

print("\nsame content, same relative distance of 5, different absolute positions:")
for pq, pk in [(5, 10), (50, 55), (150, 155), (200, 205)]:
    print(f"   positions ({pq:3d}, {pk:3d})  score {score_at(pq, pk):+.6f}")
print("\nIdentical. The score depends on (m - n), not on m and n separately.")

print("\ndifferent relative distances:")
for dist in [1, 2, 5, 20]:
    print(f"   distance {dist:2d}  score {score_at(50, 50 + dist):+.6f}")

rope cache: (256, 64) - zero learned parameters

same content, same relative distance of 5, different absolute positions:
   positions (  5,  10)  score -2.752194
   positions ( 50,  55)  score -2.752197
   positions (150, 155)  score -2.752192
   positions (200, 205)  score -2.752203

Identical. The score depends on (m - n), not on m and n separately.

different relative distances:
   distance  1  score +0.452605
   distance  2  score +2.906136
   distance  5  score -2.752197
   distance 20  score -1.052121


## 4. Grouped-Query Attention

Several query heads share one key/value head. The KV cache shrinks by the group factor.

In [5]:
def repeat_kv(x, n_rep):
    b, n_kv, seq, hd = x.shape
    if n_rep == 1:
        return x
    return x[:, :, None].expand(b, n_kv, n_rep, seq, hd).reshape(b, n_kv * n_rep, seq, hd)


class GroupedQueryAttention(nn.Module):
    def __init__(self, d, n_heads, n_kv_heads):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = d // n_heads

        self.W_q = nn.Linear(d, n_heads * self.head_dim, bias=False)
        self.W_k = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)   # SMALLER
        self.W_v = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)   # SMALLER
        self.out = nn.Linear(d, d, bias=False)

    def forward(self, x):
        b, n, d = x.shape
        q = self.W_q(x).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(x).view(b, n, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(x).view(b, n, self.n_kv_heads, self.head_dim).transpose(1, 2)

        k, v = repeat_kv(k, self.n_rep), repeat_kv(v, self.n_rep)   # expand on the fly
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        return self.out(o.transpose(1, 2).contiguous().view(b, n, d))


d, n_heads = 1024, 16
print(f"{'scheme':<10} {'KV heads':>9} {'params':>10} {'cache/token':>13} {'saving':>8}")
print("-" * 55)
base_cache = None
for label, kv in [("MHA", 16), ("GQA-8", 8), ("GQA-4", 4), ("MQA", 1)]:
    m = GroupedQueryAttention(d, n_heads, kv)
    p = sum(x.numel() for x in m.parameters())
    cache = 2 * kv * (d // n_heads) * 2          # K and V, bf16
    base_cache = base_cache or cache
    print(f"{label:<10} {kv:>9} {p:>10,} {cache:>10} B {base_cache/cache:>7.0f}x")
    out = m(torch.randn(2, 16, d))

print("\nAll produce identical output shapes; only the cache size differs.")

scheme      KV heads     params   cache/token   saving
-------------------------------------------------------
MHA               16  4,194,304       4096 B       1x
GQA-8              8  3,145,728       2048 B       2x
GQA-4              4  2,621,440       1024 B       4x
MQA                1  2,228,224        256 B      16x

All produce identical output shapes; only the cache size differs.


## 5. Mixture of Experts

Replace one FFN with N experts plus a router. Each token uses only top-k experts.

**The trap:** MoE saves *compute*, not *memory*. All experts must be resident in
VRAM even though most are idle for any given token.

In [6]:
class MoEFeedForward(nn.Module):
    def __init__(self, d, n_experts=8, top_k=2):
        super().__init__()
        self.n_experts, self.top_k = n_experts, top_k
        self.router = nn.Linear(d, n_experts, bias=True)   # the bias is the balancing knob
        nn.init.zeros_(self.router.bias)
        self.experts = nn.ModuleList([SwiGLU(d) for _ in range(n_experts)])
        self.last_counts = None

    def forward(self, x):
        b, seq, d = x.shape
        flat = x.view(-1, d)

        probs = self.router(flat).softmax(-1)
        weights, indices = torch.topk(probs, self.top_k, dim=-1)
        weights = weights / weights.sum(-1, keepdim=True)

        self.last_counts = torch.bincount(indices.flatten(), minlength=self.n_experts)

        out = torch.zeros_like(flat)
        for slot in range(self.top_k):
            for e_id, expert in enumerate(self.experts):
                mask = indices[:, slot] == e_id
                if mask.any():
                    out[mask] += weights[mask, slot, None] * expert(flat[mask])
        return out.view(b, seq, d)


d = 256
dense = SwiGLU(d)
moe = MoEFeedForward(d, n_experts=8, top_k=2)

dense_p = sum(p.numel() for p in dense.parameters())
moe_total = sum(p.numel() for p in moe.parameters())
moe_active = sum(p.numel() for p in moe.experts[0].parameters()) * 2

print(f"dense FFN parameters       : {dense_p:,}")
print(f"MoE TOTAL parameters       : {moe_total:,}   <-- all must fit in VRAM")
print(f"MoE ACTIVE per token       : {moe_active:,}   <-- only these compute")
print(f"\ncapacity multiplier: {moe_total/dense_p:.1f}x  at {moe_active/dense_p:.1f}x the per-token compute")

x = torch.randn(4, 32, d)
print("\noutput shape:", tuple(moe(x).shape))

dense FFN parameters       : 523,776
MoE TOTAL parameters       : 4,192,264   <-- all must fit in VRAM
MoE ACTIVE per token       : 1,047,552   <-- only these compute

capacity multiplier: 8.0x  at 2.0x the per-token compute



output shape: (4, 32, 256)


### Watch routing collapse

An untrained router distributes reasonably. Bias it and utilization collapses onto a few experts — the failure that load balancing exists to prevent.

In [7]:
x = torch.randn(8, 64, d)
moe(x)
counts = moe.last_counts
total = counts.sum().item()

print("balanced-ish router (fresh init):")
for i, c in enumerate(counts.tolist()):
    print(f"   expert {i}: {c:5d} tokens  {c/total*100:5.1f}%  {'#' * int(c/total*100)}")

with torch.no_grad():
    moe.router.bias[0] = 50.0          # force expert 0 to always win slot 1
    moe.router.bias[3] = 40.0          # and expert 3 to always win slot 2
moe(x)
counts = moe.last_counts
total = counts.sum().item()

print("\nafter biasing the router (simulating collapse):")
for i, c in enumerate(counts.tolist()):
    print(f"   expert {i}: {c:5d} tokens  {c/total*100:5.1f}%  {'#' * int(c/total*100)}")

print("\nTwo experts now absorb everything and six are dead weight: they occupy")
print("VRAM but never compute. (With top_k=2, two experts is the floor - one")
print("cannot exceed 50%.) Real systems prevent this with an auxiliary balancing")
print("loss, or by adjusting router biases directly so the main objective is")
print("left undistorted.")

print("\nThat router bias is not just a demo device: adjusting exactly this bias")
print("is how modern 'auxiliary-loss-free' load balancing works. Overused experts")
print("get their bias nudged DOWN and starved ones nudged UP, which rebalances")
print("routing without adding a term to the loss that would distort the main")
print("training objective.")

balanced-ish router (fresh init):
   expert 0:   135 tokens   13.2%  #############
   expert 1:   128 tokens   12.5%  ############
   expert 2:   118 tokens   11.5%  ###########
   expert 3:   127 tokens   12.4%  ############
   expert 4:   126 tokens   12.3%  ############
   expert 5:   118 tokens   11.5%  ###########
   expert 6:   131 tokens   12.8%  ############
   expert 7:   141 tokens   13.8%  #############

after biasing the router (simulating collapse):
   expert 0:   512 tokens   50.0%  ##################################################
   expert 1:     0 tokens    0.0%  
   expert 2:     0 tokens    0.0%  
   expert 3:   512 tokens   50.0%  ##################################################
   expert 4:     0 tokens    0.0%  
   expert 5:     0 tokens    0.0%  
   expert 6:     0 tokens    0.0%  
   expert 7:     0 tokens    0.0%  

Two experts now absorb everything and six are dead weight: they occupy
VRAM but never compute. (With top_k=2, two experts is the floor - one
can

## 6. Putting it together: a modern block

In [8]:
class ModernBlock(nn.Module):
    """RMSNorm + GQA + RoPE + SwiGLU, pre-norm, no biases, no dropout."""
    def __init__(self, d, n_heads, n_kv_heads, max_len=2048):
        super().__init__()
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.head_dim = d // n_heads

        self.norm1, self.norm2 = RMSNorm(d), RMSNorm(d)
        self.W_q = nn.Linear(d, n_heads * self.head_dim, bias=False)
        self.W_k = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.W_v = nn.Linear(d, n_kv_heads * self.head_dim, bias=False)
        self.out = nn.Linear(d, d, bias=False)
        self.ff = SwiGLU(d)

        cos, sin = build_rope_cache(self.head_dim, max_len)
        self.register_buffer("cos", cos)
        self.register_buffer("sin", sin)

    def forward(self, x):
        b, n, d = x.shape
        h = self.norm1(x)
        q = self.W_q(h).view(b, n, self.n_heads, self.head_dim).transpose(1, 2)
        k = self.W_k(h).view(b, n, self.n_kv_heads, self.head_dim).transpose(1, 2)
        v = self.W_v(h).view(b, n, self.n_kv_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(q, self.cos, self.sin)          # position enters HERE, not at the embedding
        k = apply_rope(k, self.cos, self.sin)

        k, v = repeat_kv(k, self.n_rep), repeat_kv(v, self.n_rep)
        o = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        x = x + self.out(o.transpose(1, 2).contiguous().view(b, n, d))
        return x + self.ff(self.norm2(x))


block = ModernBlock(d=512, n_heads=8, n_kv_heads=2)
x = torch.randn(2, 128, 512)
print("output:", tuple(block(x).shape))
print(f"parameters: {sum(p.numel() for p in block.parameters()):,}")
print()
print("This block is architecturally current: RMSNorm, GQA (4 query heads per")
print("KV head), RoPE, SwiGLU, pre-norm, no biases, no dropout.")
print("Compare it with the GPT-2 block in notebook 03 - every difference is")
print("driven by either compute cost or KV-cache pressure.")

output: (2, 128, 512)
parameters: 2,753,024

This block is architecturally current: RMSNorm, GQA (4 query heads per
KV head), RoPE, SwiGLU, pre-norm, no biases, no dropout.
Compare it with the GPT-2 block in notebook 03 - every difference is
driven by either compute cost or KV-cache pressure.


## Exercises

1. Build a full model from `ModernBlock` and compare parameter count and speed against the GPT-2 block from notebook 03 at matched width and depth.
2. Add a **shared expert** to the MoE that every token always uses. Does routing balance improve?
3. Implement **sliding-window attention** (attend only to the last 256 positions) and measure the memory saving at 4k context.
4. Implement an auxiliary load-balancing loss for the MoE router and verify it prevents the collapse shown above.

**Back to:** [Guide index](../README.md)